# Historical EDA: superseded by the revised analysis

**Revision note (2026-09-18): Do not reuse this notebook's saved plots as current thesis results.** Current descriptive analysis is in `src/descriptive_analysis.py`, with aggregate tables in `outputs/tables/` and figures generated by `src/result_figures.py`. The current primary outcome is broad abandonment (LWBS + ELOPED + LEFT AGAINST MEDICAL ADVICE), and the primary cohort retains unknown transport.

This notebook preserves the earlier exploratory work. Its primary-LWBS labels and total-patient-visit counts differ from the final outcome and strictly earlier visit predictor. Section 5's congestion exploration is abandoned: patient-specific date shifts do not reconstruct simultaneous hospital arrivals, and occupied-hour rows were not a complete clock-hour series. Calendar-time interpretations below are not used in the thesis. There is no implemented age analysis. See `IMPLEMENTATION_TASKS.md` and the current project summary for the replacement workflow.

## 1. Dataset Inventory

This first step checks the basic structure of the MIMIC-IV-ED dataset before we define any outcome or build any model. The goal is to see which ED tables are available, how many rows each table has, and which columns may be useful for the thesis. This matters because patient abandonment is a visit-level outcome, so we first need to identify the table that contains ED visits, dispositions, triage information, and possible early predictors.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

candidate_ed_dirs = [DATA_ROOT / "mimic-iv-ed-2.2" / "ed"]

ED_DIR = next((path for path in candidate_ed_dirs if path.exists()), None)

tables = [
    "diagnosis",
    "edstays",
    "medrecon",
    "pyxis",
    "triage",
    "vitalsign",
]


def find_table_path(table_name):
    for suffix in [".csv.gz", ".csv"]:
        path = ED_DIR / f"{table_name}{suffix}"
        if path.exists():
            return path
    return None


def count_rows(path, chunksize=200_000):
    return sum(len(chunk) for chunk in pd.read_csv(path, usecols=[0], chunksize=chunksize))


inventory_rows = []

for table_name in tables:
    table_path = find_table_path(table_name)
    columns = pd.read_csv(table_path, nrows=0).columns.tolist()
    inventory_rows.append(
        {
            "table": table_name,
            "row_count": count_rows(table_path),
            "column_count": len(columns),
            "columns_preview": ", ".join(columns[:12]),
        }
    )

dataset_inventory = pd.DataFrame(inventory_rows)
dataset_inventory

## 2. Disposition and Outcome Definition

The next step is to inspect the `disposition` column in `edstays`. This column describes how each ED visit ended, so it is the basis for defining patient abandonment. For the thesis, the primary abandonment outcome is `LEFT WITHOUT BEING SEEN`. A broader secondary outcome can also include `ELOPED` and `LEFT AGAINST MEDICAL ADVICE`. We inspect the disposition values first because the outcome definition should come from the actual labels in the dataset, not from assumptions.

In [ ]:
edstays_path = find_table_path("edstays")

edstays = pd.read_csv(edstays_path, low_memory=False)

print(f"Rows: {edstays.shape[0]:,}")
print(f"Columns: {edstays.shape[1]:,}")

edstays["disposition_clean"] = (
    edstays["disposition"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

disposition_counts = (
    edstays["disposition_clean"]
    .value_counts(dropna=False)
    .rename_axis("disposition")
    .reset_index(name="n")
)
disposition_counts["percent"] = (disposition_counts["n"] / len(edstays) * 100).round(2)

display(disposition_counts)

primary_abandonment = {"LEFT WITHOUT BEING SEEN"}
broad_abandonment = {
    "LEFT WITHOUT BEING SEEN",
    "ELOPED",
    "LEFT AGAINST MEDICAL ADVICE",
}

edstays["abandoned_primary_lwbs"] = edstays["disposition_clean"].isin(primary_abandonment)
edstays["abandoned_broad"] = edstays["disposition_clean"].isin(broad_abandonment)

outcome_summary = pd.DataFrame(
    [
        {
            "outcome": "Primary abandonment",
            "definition": "LEFT WITHOUT BEING SEEN",
            "n": int(edstays["abandoned_primary_lwbs"].sum()),
            "percent_of_ed_visits": round(edstays["abandoned_primary_lwbs"].mean() * 100, 2),
        },
        {
            "outcome": "Broad abandonment",
            "definition": "LEFT WITHOUT BEING SEEN + ELOPED + LEFT AGAINST MEDICAL ADVICE",
            "n": int(edstays["abandoned_broad"].sum()),
            "percent_of_ed_visits": round(edstays["abandoned_broad"].mean() * 100, 2),
        },
    ]
)

outcome_summary

## 3. Basic Abandonment Patterns

This step checks whether the primary abandonment outcome differs across simple early-visit variables. We use arrival hour from `edstays`, triage acuity from `triage`, and arrival transport from `edstays`. The `UNKNOWN` transport row is excluded from the transport summary because it does not have a clear operational interpretation.

In [ ]:
import matplotlib.pyplot as plt

triage_path = find_table_path("triage")

triage = pd.read_csv(triage_path, low_memory=False)

visits = edstays.merge(
    triage[["stay_id", "acuity"]],
    on="stay_id",
    how="left",
)

visits["intime"] = pd.to_datetime(visits["intime"], errors="coerce")
visits["arrival_hour"] = visits["intime"].dt.hour

pattern_columns = [
    "arrival_hour",
    "acuity",
    "arrival_transport",
    "abandoned_primary_lwbs",
]


def summarize_lwbs_by_group(data, group_column, min_visits=30):
    summary = (
        data.dropna(subset=[group_column])
        .groupby(group_column)
        .agg(
            ed_visits=("abandoned_primary_lwbs", "size"),
            lwbs_visits=("abandoned_primary_lwbs", "sum"),
            lwbs_rate=("abandoned_primary_lwbs", "mean"),
        )
        .reset_index()
    )
    summary = summary[summary["ed_visits"] >= min_visits].copy()
    summary["lwbs_rate_percent"] = (summary["lwbs_rate"] * 100).round(2)
    return summary.drop(columns="lwbs_rate")

lwbs_by_acuity = summarize_lwbs_by_group(visits, "acuity", min_visits=100).sort_values("acuity")
visits_known_transport = visits[visits["arrival_transport"] != "UNKNOWN"].copy()
lwbs_by_arrival_transport = summarize_lwbs_by_group(
    visits_known_transport,
    "arrival_transport",
    min_visits=100,
).sort_values(
    "lwbs_rate_percent",
    ascending=False,
)

print("LWBS by triage acuity")
display(lwbs_by_acuity)

print("LWBS by arrival transport")
display(lwbs_by_arrival_transport)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(
    lwbs_by_arrival_transport["arrival_transport"],
    lwbs_by_arrival_transport["lwbs_rate_percent"],
)
axes[0].set_title("LWBS rate by arrival transport")
axes[0].set_xlabel("Arrival Transport")
axes[0].set_ylabel("LWBS rate (%)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(
    lwbs_by_acuity["acuity"].astype(str),
    lwbs_by_acuity["lwbs_rate_percent"],
)
axes[1].set_title("LWBS rate by triage acuity")
axes[1].set_xlabel("Acuity")
axes[1].set_ylabel("LWBS rate (%)")

plt.tight_layout()
plt.show()

## 4. ED Length of Stay by Disposition

This step describes how long visits lasted from ED arrival (`intime`) to ED departure (`outtime`). It helps compare abandonment-related dispositions with completed ED care pathways using both a summary table and plots. This variable is descriptive for now: final ED length of stay is known only after the visit ends, so it should not be used as a predictor unless the prediction moment is defined differently later.

In [ ]:
edstays["intime"] = pd.to_datetime(edstays["intime"], errors="coerce")
edstays["outtime"] = pd.to_datetime(edstays["outtime"], errors="coerce")

edstays["ed_los_minutes"] = (
    edstays["outtime"] - edstays["intime"]
).dt.total_seconds() / 60

valid_los = edstays[
    edstays["ed_los_minutes"].notna()
    & (edstays["ed_los_minutes"] >= 0)
].copy()

los_by_disposition = (
    valid_los.groupby("disposition_clean")
    .agg(
        ed_visits=("stay_id", "size"),
        median_los_minutes=("ed_los_minutes", "median"),
        q1_los_minutes=("ed_los_minutes", lambda x: x.quantile(0.25)),
        q3_los_minutes=("ed_los_minutes", lambda x: x.quantile(0.75)),
        mean_los_minutes=("ed_los_minutes", "mean"),
    )
    .reset_index()
)

los_by_disposition["median_los_hours"] = los_by_disposition["median_los_minutes"] / 60

number_columns = [
    "median_los_minutes",
    "q1_los_minutes",
    "q3_los_minutes",
    "mean_los_minutes",
    "median_los_hours",
]
los_by_disposition[number_columns] = los_by_disposition[number_columns].round(2)

los_by_disposition = los_by_disposition.sort_values("median_los_minutes")
display(los_by_disposition)

plt.figure(figsize=(10, 5))
plt.barh(
    los_by_disposition["disposition_clean"],
    los_by_disposition["median_los_minutes"],
)
plt.title("Median ED length of stay by disposition")
plt.xlabel("Median ED length of stay (minutes)")
plt.ylabel("Disposition")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

los_boxplot_data = valid_los[["disposition_clean", "ed_los_minutes"]].dropna().copy()

los_plot_limit = los_boxplot_data["ed_los_minutes"].quantile(0.99)
los_boxplot_data = los_boxplot_data[
    los_boxplot_data["ed_los_minutes"] <= los_plot_limit
].copy()

disposition_order = (
    los_boxplot_data.groupby("disposition_clean")["ed_los_minutes"]
    .median()
    .sort_values()
    .index
    .tolist()
)

boxplot_values = [
    los_boxplot_data.loc[
        los_boxplot_data["disposition_clean"] == disposition,
        "ed_los_minutes",
    ]
    for disposition in disposition_order
]

plt.figure(figsize=(12, 5))
plt.boxplot(boxplot_values, tick_labels=disposition_order, showfliers=False)
plt.title("ED length of stay distribution by disposition")
plt.xlabel("Disposition")
plt.ylabel("ED length of stay (minutes)")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Simple Congestion Proxy EDA

This step creates simple arrival-volume proxies for ED congestion. The idea is that if many patients arrived recently, the ED may be busier and patients may be more likely to leave without being seen.

The comparison below identifies the top 10% busiest recent-arrival periods and compares them with all other periods. This is more stable than using the single busiest hour, because it keeps enough visits for the LWBS rate to be meaningful. These are still rough proxies, not true queue length measures, but they connect the EDA to ED crowding and queue abandonment theory.

In [ ]:
congestion_eda = edstays.copy()
congestion_eda["intime"] = pd.to_datetime(congestion_eda["intime"], errors="coerce")
congestion_eda["arrival_hour_block"] = congestion_eda["intime"].dt.floor("h")

hourly_arrivals = (
    congestion_eda.dropna(subset=["arrival_hour_block"])
    .groupby("arrival_hour_block")
    .size()
    .reset_index(name="same_hour_arrivals")
    .sort_values("arrival_hour_block")
)

hourly_arrivals["previous_1h_arrivals"] = hourly_arrivals["same_hour_arrivals"].shift(1)
hourly_arrivals["previous_2h_arrivals"] = (
    hourly_arrivals["same_hour_arrivals"]
    .shift(1)
    .rolling(window=2, min_periods=1)
    .sum()
)

congestion_eda = congestion_eda.merge(
    hourly_arrivals,
    on="arrival_hour_block",
    how="left",
)

previous_1h_top_cutoff = hourly_arrivals["previous_1h_arrivals"].dropna().quantile(0.90)
previous_2h_top_cutoff = hourly_arrivals["previous_2h_arrivals"].dropna().quantile(0.90)

peak_cutoff_summary = pd.DataFrame(
    [
        {
            "proxy": "Previous 1-hour arrivals",
            "top_10_percent_cutoff": previous_1h_top_cutoff,
            "max_arrivals": hourly_arrivals["previous_1h_arrivals"].max(),
        },
        {
            "proxy": "Previous 2-hour arrivals",
            "top_10_percent_cutoff": previous_2h_top_cutoff,
            "max_arrivals": hourly_arrivals["previous_2h_arrivals"].max(),
        },
    ]
)

display(peak_cutoff_summary)

congestion_eda["previous_1h_peak_group"] = "Other periods"
congestion_eda.loc[
    congestion_eda["previous_1h_arrivals"] >= previous_1h_top_cutoff,
    "previous_1h_peak_group",
] = "Top 10% busiest periods"
congestion_eda.loc[congestion_eda["previous_1h_arrivals"].isna(), "previous_1h_peak_group"] = pd.NA

congestion_eda["previous_2h_peak_group"] = "Other periods"
congestion_eda.loc[
    congestion_eda["previous_2h_arrivals"] >= previous_2h_top_cutoff,
    "previous_2h_peak_group",
] = "Top 10% busiest periods"
congestion_eda.loc[congestion_eda["previous_2h_arrivals"].isna(), "previous_2h_peak_group"] = pd.NA

def summarize_lwbs_by_peak_group(data, group_column, count_column):
    summary = (
        data.dropna(subset=[group_column, count_column])
        .groupby(group_column)
        .agg(
            ed_visits=("abandoned_primary_lwbs", "size"),
            lwbs_visits=("abandoned_primary_lwbs", "sum"),
            lwbs_rate=("abandoned_primary_lwbs", "mean"),
            median_arrivals=(count_column, "median"),
            min_arrivals=(count_column, "min"),
            max_arrivals=(count_column, "max"),
        )
        .reindex(["Other periods", "Top 10% busiest periods"])
        .reset_index()
    )
    summary["lwbs_rate_percent"] = (summary["lwbs_rate"] * 100).round(2)
    return summary.drop(columns="lwbs_rate")


lwbs_by_previous_1h_peak = summarize_lwbs_by_peak_group(
    congestion_eda,
    "previous_1h_peak_group",
    "previous_1h_arrivals",
)
lwbs_by_previous_2h_peak = summarize_lwbs_by_peak_group(
    congestion_eda,
    "previous_2h_peak_group",
    "previous_2h_arrivals",
)

print("LWBS in top 10% busiest previous 1-hour periods")
display(lwbs_by_previous_1h_peak)

print("LWBS in top 10% busiest previous 2-hour periods")
display(lwbs_by_previous_2h_peak)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    lwbs_by_previous_1h_peak["previous_1h_peak_group"],
    lwbs_by_previous_1h_peak["lwbs_rate_percent"],
)
axes[0].set_title("LWBS rate: previous 1-hour peak periods")
axes[0].set_xlabel("Recent-arrival period group")
axes[0].set_ylabel("LWBS rate (%)")

axes[1].bar(
    lwbs_by_previous_2h_peak["previous_2h_peak_group"],
    lwbs_by_previous_2h_peak["lwbs_rate_percent"],
)
axes[1].set_title("LWBS rate: previous 2-hour peak periods")
axes[1].set_xlabel("Recent-arrival period group")
axes[1].set_ylabel("LWBS rate (%)")

for axis in axes:
    axis.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 6. Arrival Time Patterns

This step expands the arrival-time analysis with a stronger focus on hour-of-day patterns. It keeps weekday and weekend checks, but replaces the month trend with detailed arrival-hour and broad time-period summaries. This is more useful for the thesis because midnight and late-night arrivals may reflect different waiting conditions, staffing patterns, and patient behavior.

In [ ]:
time_patterns = edstays.copy()
time_patterns["intime"] = pd.to_datetime(time_patterns["intime"], errors="coerce")

time_patterns["arrival_weekday"] = time_patterns["intime"].dt.dayofweek
time_patterns["arrival_weekday_name"] = time_patterns["intime"].dt.day_name()
time_patterns["arrival_day_type"] = "Weekday"
time_patterns.loc[time_patterns["arrival_weekday"] >= 5, "arrival_day_type"] = "Weekend"
time_patterns["arrival_hour"] = time_patterns["intime"].dt.hour
time_patterns["arrival_time_period"] = pd.NA
time_patterns.loc[time_patterns["arrival_hour"].between(0, 5), "arrival_time_period"] = "Night (00-05)"
time_patterns.loc[time_patterns["arrival_hour"].between(6, 11), "arrival_time_period"] = "Morning (06-11)"
time_patterns.loc[time_patterns["arrival_hour"].between(12, 17), "arrival_time_period"] = "Afternoon (12-17)"
time_patterns.loc[time_patterns["arrival_hour"].between(18, 23), "arrival_time_period"] = "Evening (18-23)"

time_columns = [
    "intime",
    "arrival_weekday_name",
    "arrival_day_type",
    "arrival_hour",
    "arrival_time_period",
]

def summarize_lwbs_by_time_group(data, group_column, order=None, min_visits=100):
    summary = (
        data.dropna(subset=[group_column])
        .groupby(group_column)
        .agg(
            ed_visits=("abandoned_primary_lwbs", "size"),
            lwbs_visits=("abandoned_primary_lwbs", "sum"),
            lwbs_rate=("abandoned_primary_lwbs", "mean"),
        )
        .reset_index()
    )
    summary = summary[summary["ed_visits"] >= min_visits].copy()
    summary["lwbs_rate_percent"] = (summary["lwbs_rate"] * 100).round(2)
    summary = summary.drop(columns="lwbs_rate")

    if order is not None:
        summary = summary.set_index(group_column).reindex(order).reset_index()

    return summary


weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

lwbs_by_weekday = summarize_lwbs_by_time_group(
    time_patterns,
    "arrival_weekday_name",
    order=weekday_order,
)
lwbs_by_day_type = summarize_lwbs_by_time_group(
    time_patterns,
    "arrival_day_type",
    order=["Weekday", "Weekend"],
)
lwbs_by_arrival_hour = summarize_lwbs_by_time_group(
    time_patterns,
    "arrival_hour",
    order=list(range(24)),
)
lwbs_by_time_period = summarize_lwbs_by_time_group(
    time_patterns,
    "arrival_time_period",
    order=["Night (00-05)", "Morning (06-11)", "Afternoon (12-17)", "Evening (18-23)"],
)

print("LWBS by weekday")
display(lwbs_by_weekday)

print("LWBS by weekday/weekend")
display(lwbs_by_day_type)

print("LWBS by arrival hour")
display(lwbs_by_arrival_hour)

print("LWBS by broad time period")
display(lwbs_by_time_period)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].bar(
    lwbs_by_weekday["arrival_weekday_name"],
    lwbs_by_weekday["lwbs_rate_percent"],
)
axes[0].set_title("LWBS rate by weekday")
axes[0].set_xlabel("Weekday")
axes[0].set_ylabel("LWBS rate (%)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(
    lwbs_by_day_type["arrival_day_type"],
    lwbs_by_day_type["lwbs_rate_percent"],
)
axes[1].set_title("LWBS rate by weekday/weekend")
axes[1].set_xlabel("Day type")
axes[1].set_ylabel("LWBS rate (%)")

axes[2].plot(
    lwbs_by_arrival_hour["arrival_hour"],
    lwbs_by_arrival_hour["lwbs_rate_percent"],
    marker="o",
)
axes[2].set_title("LWBS rate by arrival hour")
axes[2].set_xlabel("Arrival hour")
axes[2].set_ylabel("LWBS rate (%)")
axes[2].set_xticks(range(0, 24))
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Triage Acuity and Arrival Transport

This step studies triage acuity together with arrival transport. The research question is whether abandonment is concentrated among lower-acuity patients and known transport pathways such as walk-in or ambulance arrival. The `UNKNOWN` transport category and missing acuity category are excluded here because they cannot support a clear service-system interpretation.

In [ ]:
acuity_transport = visits.copy()
acuity_transport["acuity_label"] = acuity_transport["acuity"].astype("string").fillna("Missing")
acuity_transport["arrival_transport_label"] = acuity_transport["arrival_transport"].fillna("Missing")

unknown_transport_count = (acuity_transport["arrival_transport_label"] == "UNKNOWN").sum()
missing_acuity_count = (acuity_transport["acuity_label"] == "Missing").sum()

acuity_transport = acuity_transport[
    (acuity_transport["arrival_transport_label"] != "UNKNOWN")
    & (acuity_transport["acuity_label"] != "Missing")
].copy()

acuity_transport_summary = (
    acuity_transport
    .groupby(["acuity_label", "arrival_transport_label"])
    .agg(
        ed_visits=("abandoned_primary_lwbs", "size"),
        lwbs_visits=("abandoned_primary_lwbs", "sum"),
        lwbs_rate=("abandoned_primary_lwbs", "mean"),
    )
    .reset_index()
)

acuity_transport_summary = acuity_transport_summary[acuity_transport_summary["ed_visits"] >= 100].copy()
acuity_transport_summary["lwbs_rate_percent"] = (
    acuity_transport_summary["lwbs_rate"] * 100
).round(2)
acuity_transport_summary = acuity_transport_summary.drop(columns="lwbs_rate")

print("Acuity and transport combinations with at least 100 ED visits")
display(
    acuity_transport_summary.sort_values(
        ["acuity_label", "lwbs_rate_percent"],
        ascending=[True, False],
    )
)

transport_order = (
    acuity_transport_summary
    .groupby("arrival_transport_label")["ed_visits"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)
acuity_order = sorted(acuity_transport_summary["acuity_label"].unique())

acuity_transport_pivot = acuity_transport_summary.pivot(
    index="acuity_label",
    columns="arrival_transport_label",
    values="lwbs_rate_percent",
).reindex(index=acuity_order, columns=transport_order)

display(acuity_transport_pivot)

fig, ax = plt.subplots(figsize=(10, 4))
heatmap = ax.imshow(acuity_transport_pivot, aspect="auto", cmap="Blues")

ax.set_title("LWBS rate by acuity and arrival transport")
ax.set_xlabel("Arrival transport")
ax.set_ylabel("Triage acuity")
ax.set_xticks(range(len(acuity_transport_pivot.columns)))
ax.set_xticklabels(acuity_transport_pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(acuity_transport_pivot.index)))
ax.set_yticklabels(acuity_transport_pivot.index)

for row_index, acuity_value in enumerate(acuity_transport_pivot.index):
    for column_index, transport_value in enumerate(acuity_transport_pivot.columns):
        value = acuity_transport_pivot.loc[acuity_value, transport_value]
        if pd.notna(value):
            ax.text(column_index, row_index, f"{value:.2f}", ha="center", va="center")

plt.colorbar(heatmap, ax=ax, label="LWBS rate (%)")
plt.tight_layout()
plt.show()

## 8. Repeat-Patient EDA

This step checks whether abandonment is related to patients who appear multiple times in the ED data. It summarizes LWBS by the patient's total number of ED visits in this dataset. This is useful for research interpretation and for later modeling decisions, because repeated visits from the same patient may affect how we split training and test data.

In [ ]:
patient_visit_counts = (
    edstays
    .groupby("subject_id")
    .agg(
        total_ed_visits=("stay_id", "size"),
        lwbs_visits=("abandoned_primary_lwbs", "sum"),
        broad_abandonment_visits=("abandoned_broad", "sum"),
    )
    .reset_index()
)

patient_visit_counts["repeat_patient"] = patient_visit_counts["total_ed_visits"] > 1
patient_visit_counts["ever_lwbs"] = patient_visit_counts["lwbs_visits"] > 0

patient_level_summary = pd.DataFrame(
    [
        {
            "patients": patient_visit_counts["subject_id"].nunique(),
            "repeat_patients": int(patient_visit_counts["repeat_patient"].sum()),
            "repeat_patients_percent": round(patient_visit_counts["repeat_patient"].mean() * 100, 2),
            "patients_with_lwbs": int(patient_visit_counts["ever_lwbs"].sum()),
            "patients_with_lwbs_percent": round(patient_visit_counts["ever_lwbs"].mean() * 100, 2),
        }
    ]
)

display(patient_level_summary)

visit_repeat_eda = edstays.merge(
    patient_visit_counts[["subject_id", "total_ed_visits", "repeat_patient"]],
    on="subject_id",
    how="left",
)

visit_repeat_eda["patient_visit_frequency_group"] = "11+ visits"
visit_repeat_eda.loc[visit_repeat_eda["total_ed_visits"] == 1, "patient_visit_frequency_group"] = "1 visit"
visit_repeat_eda.loc[
    visit_repeat_eda["total_ed_visits"].between(2, 3),
    "patient_visit_frequency_group",
] = "2-3 visits"
visit_repeat_eda.loc[
    visit_repeat_eda["total_ed_visits"].between(4, 10),
    "patient_visit_frequency_group",
] = "4-10 visits"

frequency_order = ["1 visit", "2-3 visits", "4-10 visits", "11+ visits"]

repeat_visit_summary = (
    visit_repeat_eda
    .groupby("patient_visit_frequency_group")
    .agg(
        patients=("subject_id", "nunique"),
        ed_visits=("stay_id", "size"),
        lwbs_visits=("abandoned_primary_lwbs", "sum"),
        lwbs_rate=("abandoned_primary_lwbs", "mean"),
        median_total_ed_visits=("total_ed_visits", "median"),
    )
    .reindex(frequency_order)
    .reset_index()
)
repeat_visit_summary["lwbs_rate_percent"] = (repeat_visit_summary["lwbs_rate"] * 100).round(2)
repeat_visit_summary = repeat_visit_summary.drop(columns="lwbs_rate")

display(repeat_visit_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    repeat_visit_summary["patient_visit_frequency_group"],
    repeat_visit_summary["lwbs_rate_percent"],
)
ax.set_title("LWBS rate by patient ED visit frequency")
ax.set_xlabel("Patient visit frequency in dataset")
ax.set_ylabel("LWBS rate (%)")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 9. Sex/Gender EDA

This step compares abandonment rates by the recorded `gender` variable in `edstays`. It helps check whether male and female ED visits have different LWBS or broad abandonment rates. This is useful as descriptive EDA and later as a possible modeling feature, but it should be interpreted carefully as an association rather than a causal explanation.

In [ ]:
sex_gender_eda = edstays.copy()

gender_label_map = {
    "F": "Female",
    "M": "Male",
}

sex_gender_eda["sex_gender_label"] = sex_gender_eda["gender"].map(gender_label_map)
sex_gender_eda["sex_gender_label"] = sex_gender_eda["sex_gender_label"].fillna(sex_gender_eda["gender"])
sex_gender_eda["sex_gender_label"] = sex_gender_eda["sex_gender_label"].fillna("Missing")

sex_gender_summary = (
    sex_gender_eda
    .groupby("sex_gender_label")
    .agg(
        ed_visits=("stay_id", "size"),
        lwbs_visits=("abandoned_primary_lwbs", "sum"),
        broad_abandonment_visits=("abandoned_broad", "sum"),
        lwbs_rate=("abandoned_primary_lwbs", "mean"),
        broad_abandonment_rate=("abandoned_broad", "mean"),
    )
    .reset_index()
)

sex_gender_summary["lwbs_rate_percent"] = (sex_gender_summary["lwbs_rate"] * 100).round(2)
sex_gender_summary["broad_abandonment_rate_percent"] = (
    sex_gender_summary["broad_abandonment_rate"] * 100
).round(2)
sex_gender_summary = sex_gender_summary.drop(columns=["lwbs_rate", "broad_abandonment_rate"])

display(sex_gender_summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    sex_gender_summary["sex_gender_label"],
    sex_gender_summary["lwbs_rate_percent"],
)
ax.set_title("LWBS rate by recorded sex/gender")
ax.set_xlabel("Recorded sex/gender")
ax.set_ylabel("LWBS rate (%)")

plt.tight_layout()
plt.show()